# Deep Learning 基礎講座 最終課題: VQA

画像と質問から回答を予測するタスク。
本 Notebook は学習〜推論〜提出ファイル作成までを 1 ファイルにまとめた提出用ファイルです。

実行すると以下を生成します。
- `submission.npy` : テストデータ(valid.json)に対する予測
- `model.pt` : 予測に使用したモデルの重み
- `submission.zip` : 上記 + 本 Notebook をまとめた提出用 zip


## 1. import

In [ ]:
import os
import re
import time
import random
from statistics import mode
from zipfile import ZipFile

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms


## 2. 設定 (config)

実験設定はここを変更する。

In [ ]:
SEED = 42

BATCH_SIZE = 128

NUM_EPOCHS = 25

PATIENCE = 3

# 最終提出用: True で train_split + valid_split を結合して全データ学習する。
# valid が無いので early-stopping は使えず、FINAL_EPOCHS 分だけ固定で回す。
# 運用: 開発(クリーンholdout)で early-stopping した epoch 数を FINAL_EPOCHS に入れ、
#       TRAIN_ON_ALL=True で再学習 → best_model.pt を提出に使う。
TRAIN_ON_ALL = False
FINAL_EPOCHS = 10

LR = 1e-4

WEIGHT_DECAY = 1e-5

OPTIMIZER = "adam"

LOSS_TYPE = "hard"

# True: 正解集計から "unanswerable" を除外する
#   - hard: mode_answer を unanswerable 以外で取る
#   - soft: soft target から unanswerable を除外する
# False: unanswerable も通常ラベルとして扱う
EXCLUDE_UNANSWERABLE = False

# クラス重み: 頻出ラベルの loss を下げて過適応（量産）を防ぐ。{ラベル: 重み}。
# 未指定ラベルは 1.0。空 dict {} で無効（全クラス 1.0）。hard/soft 両対応。
# 例: unanswerable/yes/no が多すぎるので学習時のペナルティを軽くする。
# <unk> は希少回答の寄せ集めで最大の「ゴミ箱クラス」。重みを下げて、
# 予測が <unk>/unanswerable の2クラスに崩壊するのを防ぐ。
CLASS_WEIGHTS = {
    "<unk>": 0.3,
    "unanswerable": 0.8,
    "yes": 3,
    "no": 3,
}

# 推論時、質問タイプ別に unanswerable の logit に「足す」値（学習は不変・推論だけ調整）。
#   値 > 0: そのタイプで unanswerable を出しやすく（answerable率が低い count 等）
#   値 < 0: 出しにくく（answerable率が高い color 等）
# タイプ分類は src.utils.question_type（color/count/yes-no/what(other)/...）。
# 未指定タイプには UNANSWERABLE_BIAS_DEFAULT を使う。空 dict + default 0.0 で無効。
# 最適値は valid で自動探索:  python -m src.sweep_unanswerable
#   （タイプ別に honest acc を最大化する bias を出し、この dict 形式で表示する）
UNANSWERABLE_BIAS_BY_QTYPE = {
    # 例（sweep の結果を貼る）:
    # "count": 3.0,    # ほぼ unanswerable が正解
    # "color": -1.5,   # 画像で答えられるので unanswerable に逃がさない
}
UNANSWERABLE_BIAS_DEFAULT = 0.0

# 回答語彙の最低出現回数。train でこの回数以上出た回答だけをクラスにする。
# それ未満の希少回答は "<unk>" にまとめる（1例しかないラベルは学習不能なため）。
# 1 で従来どおり全回答をクラス化。標準的な VQA は 8〜10 程度。
#   ≥1:40244  ≥3:7141  ≥5:4319  ≥8:2521  ≥10:1745 クラス
# min が大きいほど <unk> 行きが増える（mode が <unk>: min3=18.7% / min8=30.9%）
# 8→3 に下げて <unk> 吸い込みを 30.9%→18.7% に減らし、実ラベルを増やす
# （崩壊対策。クラス数 2521→7141 に増える）。
MIN_ANSWER_COUNT = 3

# 質問文の系列長。Embedding+LSTM のテキストエンコーダ用に、質問を
# 単語インデックス列としてこの長さに切り詰め／パディングする。
# （旧 bag-of-words 1層 Linear からの置き換え。語順を使えるようになる。）
MAX_QLEN = 20

IMAGE_SIZE = 360

# データ拡張（学習時のみ。inference/analyze には適用しない）。
# 色相(hue)は触らない: 色を答える質問が多く、hue揺らしは正解を壊すため。
AUGMENT = True
# 少数クラス対策: train 頻度がこの値以下のクラスのサンプルには、
# 通常の軽い aug ではなく「強い aug」を per-sample で適用して多様性を稼ぐ。
# 0 で無効（全データ一律の軽い aug のみ）。<unk> は対象外。
MINORITY_MAX_COUNT = 50

# 画像の正規化（ToTensor の後に適用）。train/inference/analyze で共通化される。
# True にする場合、学習・推論・分析の全てで同じ統計が使われる。
# mean/std は ImageNet 統計。スクラッチ学習なので、より厳密には
#   python -m src.compute_stats
# で train 画像から実測した値に差し替えるのが望ましい。
NORMALIZE = True
NORM_MEAN = [0.485, 0.456, 0.406]
NORM_STD = [0.229, 0.224, 0.225]

MODEL = "baseline"
EXP_NAME = "baseline"

# 画像だけ probe（src.color_probe）が対象にする質問タイプ。
#   "color" / "count" / "yes/no" / "what(other)" など（src.question_only の分類）。
# 画像だけで当てられるか＝画像に情報があるかを測る。config で対象を切替可能。
PROBE_QTYPE = "color"

# 画像と質問の融合方式（VQAModel が参照）。
#   "concat"          : 画像をavgpoolした512次元と質問特徴を連結（従来）。
#                       画像を1ベクトルに潰すので「どこを見るか」を質問で選べない。
#   "cross_attention" : 質問トークンが画像の空間特徴(H×W)に attention する。
#                       color/count など「画像の特定箇所を見る」問題向け。
# 診断（src.question_only）の結論: 質問priorは~0.59で飽和、伸びしろは画像側。
# cross_attention は画像情報を答えに効かせるための本命。
FUSION = "concat"

# #3「画像を捨てさせない学習」: 画像プール特徴だけから回答を予測する補助ヘッドを付け、
# 総ロス = 主ロス + AUX_IMAGE_LOSS_WEIGHT × 画像onlyロス。
# 画像branch に必ず勾配を流し、言語prior へのショートカット（画像無視）を防ぐ。
# 0.0 で無効。0.3〜1.0 程度から。推論には影響しない（補助ヘッドは推論で未使用）。
# 注意: train と inference で ≷0 を揃えること（アーキテクチャが変わるため）。
AUX_IMAGE_LOSS_WEIGHT = 0.0

RESNET = "resnet50"  # 画像エンコーダ: "resnet18" / "resnet34" / "resnet50"

# 自己教師あり事前学習で得たバックボーン重みのパス。None でスクラッチ。
# 生成: python -m src.ssl_pretrain  → ./outputs/checkpoints/ssl_backbone.pt
# RESNET と同じ種別の重みであること（resnet50 で SSL したら resnet50 で使う）。
PRETRAINED_BACKBONE = PRETRAINED_BACKBONE = "./outputs/checkpoints/ssl_backbone.pt"

# train.py を1回実行するだけで最後まで走らせるための自動化（inference.main が参照）。
# AUTO_SWEEP_UNANSWERABLE: 推論前に valid でタイプ別 unanswerable bias を自動探索し、
#   その結果を提出に適用する（手動 sweep→config 貼り付けが不要に）。
#   TRAIN_ON_ALL 時は valid が学習に含まれ過学習なので無効化される。
#   False のときは config の UNANSWERABLE_BIAS_BY_QTYPE をそのまま使う。
# AUTO_BUILD_NOTEBOOK: 提出zip の前に統合Notebookを自動生成（未ビルドでも走り切る）。
AUTO_SWEEP_UNANSWERABLE = True
AUTO_BUILD_NOTEBOOK = True

MODEL_PATH = "./outputs/checkpoints/best_model.pt"
NOTEBOOK_PATH = "./DL_Basic_2026_Spring_competition_VQA.ipynb"

## 3. utils

In [ ]:
    IMAGE_SIZE, NORMALIZE, NORM_MEAN, NORM_STD, AUGMENT,
)

def build_transform(train=False, strong=False):
    """画像前処理を返す。

    - train=False（既定）: Resize→ToTensor(→Normalize)。inference/analyze 用。
      推論はランダム性を入れたくないので必ずこれ。
    - train=True かつ AUGMENT=True: 全データ共通の「軽い aug」を付与。
      色を答える質問が多いので hue/saturation は揺らさない（色回答の保護）。
    - train=True かつ strong=True: 少数クラス向けの「強い aug」を上乗せ
      （ブラー・遠近変形）。VizWiz のブレ/構図崩れにも効く。

    NORMALIZE で Normalize の有無を切替。学習時と推論時で Resize/Normalize
    が共通なので分布ズレは起きない（変わるのは aug の有無のみ）。
    """
    if train and AUGMENT:
        # 軽い aug（全データ共通）
        ops = [
            transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.85, 1.0)),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2),  # hue/sat はゼロ
        ]
        if strong:
            # 少数クラス向けの追加 aug（より強い変形で多様性を稼ぐ）
            ops += [
                transforms.RandomApply(
                    [transforms.GaussianBlur(kernel_size=5)], p=0.4
                ),
                transforms.RandomPerspective(distortion_scale=0.2, p=0.4),
                transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.7, 1.0)),
            ]
    else:
        ops = [transforms.Resize((IMAGE_SIZE, IMAGE_SIZE))]

    ops.append(transforms.ToTensor())
    if NORMALIZE:
        ops.append(transforms.Normalize(mean=NORM_MEAN, std=NORM_STD))
    return transforms.Compose(ops)

def question_type(q):
    """正規化済み質問文 q を粗い意味タイプに分類する。
    質問タイプ別の unanswerable 補正（推論）や診断の集計に使う。"""
    w = q.split(" ") if q else []
    head = w[0] if w else ""
    head2 = " ".join(w[:2])
    if "color" in q or "colour" in q:
        return "color"
    if head2 in ("how many", "how much") or "how many" in q:
        return "count"
    if "brand" in q:
        return "brand"
    if head in ("is", "are", "was", "were", "do", "does", "did",
                "can", "could", "will", "would", "should", "has", "have"):
        return "yes/no"
    if head == "what":
        return "what(other)"
    if head in ("where", "who", "when", "why", "which", "how"):
        return head
    return "other"

def apply_unanswerable_bias(logits, qtexts, unanswerable_idx,
                            by_qtype, default_bias=0.0):
    """質問タイプ別に unanswerable の logit を調整する（推論時のみ。学習は不変）。

    logits[i, unanswerable_idx] += bias を行う。
      bias > 0: その質問タイプで unanswerable を出しやすく（answerable率が低い count 等）
      bias < 0: 出しにくく（answerable率が高い color 等）
    bias は by_qtype[type] を引き、無ければ default_bias。

    Parameters
    ----------
    logits : torch.Tensor  (N, C)  ※ in-place で書き換える
    qtexts : list[str]     process_text 済みの質問文（len==N）
    unanswerable_idx : int | None  None なら何もしない
    by_qtype : dict        {質問タイプ: bias}
    default_bias : float   未指定タイプに使う bias
    """
    if unanswerable_idx is None:
        return logits
    for i, q in enumerate(qtexts):
        b = by_qtype.get(question_type(q), default_bias)
        if b:
            logits[i, unanswerable_idx] += b
    return logits

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 4. データセット

In [ ]:
    EXCLUDE_UNANSWERABLE, MIN_ANSWER_COUNT, MINORITY_MAX_COUNT, MAX_QLEN,
)

UNK_ANSWER = "<unk>"

def process_text(text):
    """
    入力文と回答のフォーマットを統一するための関数．

    Parameters
    ----------
    text : str
        入力文，もしくは回答．
    """
    # lowercase
    text = text.lower()

    # 数詞を数字に変換
    num_word_to_digit = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'ten': '10'
    }
    for word, digit in num_word_to_digit.items():
        text = text.replace(word, digit)

    # 小数点のピリオドを削除
    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)

    # 冠詞の削除
    text = re.sub(r'\b(a|an|the)\b', '', text)

    # 短縮形のカンマの追加
    contractions = {
        "dont": "don't", "isnt": "isn't", "arent": "aren't", "wont": "won't",
        "cant": "can't", "wouldnt": "wouldn't", "couldnt": "couldn't"
    }
    for contraction, correct in contractions.items():
        text = text.replace(contraction, correct)

    # 句読点をスペースに変換
    text = re.sub(r"[^\w\s':]", ' ', text)

    # 句読点をスペースに変換
    text = re.sub(r'\s+,', ',', text)

    # 連続するスペースを1つに変換
    text = re.sub(r'\s+', ' ', text).strip()

    return text

class VQADataset(torch.utils.data.Dataset):
    """
    VQA データセットを扱うためのクラス．
    """
    def __init__(self, df_path, image_dir, transform=None, answer=True,
                 strong_transform=None):
        self.transform = transform  # 画像の前処理（軽い aug もしくは eval）
        self.strong_transform = strong_transform  # 少数クラス向けの強い aug（任意）
        self.image_dir = image_dir  # 画像ファイルのディレクトリ
        # df_path は単一パスでも、複数パスのリスト（全データ学習用に train+valid 結合）でも可
        if isinstance(df_path, (list, tuple)):
            self.df = pd.concat(
                [pd.read_json(p) for p in df_path],
                ignore_index=True,
            )
        else:
            self.df = pd.read_json(df_path)  # 画像ファイルのパス，question, answerを持つDataFrame
        self.answer = answer

        # question / answerの辞書を作成
        self.question2idx = {}
        self.answer2idx = {}
        self.idx2question = {}
        self.idx2answer = {}
        self.minority_answer_idx = set()  # 強い aug 対象（answer=True 時に構築）

        # 質問文に含まれる単語を辞書に追加
        for question in self.df["question"]:
            question = process_text(question)
            words = question.split(" ")
            for word in words:
                if word not in self.question2idx:
                    self.question2idx[word] = len(self.question2idx)
        self.idx2question = {v: k for k, v in self.question2idx.items()}  # 逆変換用の辞書(question)

        if self.answer:
            # 回答の出現回数を数え、MIN_ANSWER_COUNT 以上のものだけをクラス化する。
            # 希少回答（1例しかない等）は学習不能なので "<unk>" にまとめる。
            answer_counter = Counter()
            for answers in self.df["answers"]:
                for answer in answers:
                    answer_counter[process_text(answer["answer"])] += 1

            for word, count in answer_counter.items():
                if count >= MIN_ANSWER_COUNT:
                    self.answer2idx[word] = len(self.answer2idx)

            # OOV（足切りされた希少回答）受け皿
            self.answer2idx[UNK_ANSWER] = len(self.answer2idx)

            self.idx2answer = {v: k for k, v in self.answer2idx.items()}  # 逆変換用の辞書(answer)

            # 少数クラス集合（強い aug 対象）。
            # 語彙に残った（count>=MIN_ANSWER_COUNT）かつ低頻度（<=MINORITY_MAX_COUNT）のクラス。
            # <unk> は多数の希少回答の集約なので対象外。
            self.minority_answer_idx = {
                self.answer2idx[word]
                for word, count in answer_counter.items()
                if word in self.answer2idx and count <= MINORITY_MAX_COUNT
            }

    def update_dict(self, dataset):
        """
        検証用データ，テストデータの辞書を訓練データの辞書に更新する．

        Parameters
        ----------
        dataset : Dataset
            訓練データのDataset
        """
        self.question2idx = dataset.question2idx
        self.answer2idx = dataset.answer2idx
        self.idx2question = dataset.idx2question
        self.idx2answer = dataset.idx2answer

    def __getitem__(self, idx):
        """
        対応するidxのデータ（画像，質問，回答）を取得．

        Parameters
        ----------
        idx : int
            取得するデータのインデックス

        Returns
        -------
        image : torch.Tensor  (C, H, W)
            画像データ
        question : torch.Tensor  (MAX_QLEN,) long
            質問文を単語インデックス列にしたもの（Embedding+LSTM 用）。
            未知語は UNK=len(question2idx)、空き要素は PAD=len(question2idx)+1。
        answers : torch.Tensor  (n_answer)
            10人の回答者の回答のid
        mode_answer_idx : torch.Tensor  (1)
            10人の回答者の回答の中で最頻値の回答のid
        """
        # 画像は mode_answer（少数クラス判定）を求めてから transform を選ぶため、
        # ここでは PIL のまま読み込んでおく。
        image = Image.open(f"{self.image_dir}/{self.df['image'][idx]}").convert("RGB")
        # 質問を単語インデックス列にする（語順を保持。Embedding+LSTM 用）。
        # UNK = 未知語の受け皿 = V、PAD = 空き要素 = V+1（V=語彙数）。
        vocab = len(self.question2idx)
        unk_idx = vocab
        pad_idx = vocab + 1
        question_words = process_text(self.df["question"][idx]).split(" ")
        q_ids = [
            self.question2idx.get(word, unk_idx)
            for word in question_words
            if word != ""
        ][:MAX_QLEN]
        q_ids += [pad_idx] * (MAX_QLEN - len(q_ids))  # 末尾を PAD で埋める
        question = torch.tensor(q_ids, dtype=torch.long)

        if self.answer:
            answer_texts = [
                process_text(answer["answer"])
                for answer in self.df["answers"][idx]
            ]

            # 足切りされた回答は <unk> に写像
            unk_idx = self.answer2idx[UNK_ANSWER]
            answers = [
                self.answer2idx.get(answer_text, unk_idx)
                for answer_text in answer_texts
            ]

            if EXCLUDE_UNANSWERABLE:
                # unanswerable を除外して最頻値を取る（全件 unanswerable の時はフォールバック）
                filtered_answers = [
                    self.answer2idx.get(answer_text, unk_idx)
                    for answer_text in answer_texts
                    if answer_text != "unanswerable"
                ]
                mode_answer_idx = mode(filtered_answers) if filtered_answers else mode(answers)
            else:
                mode_answer_idx = mode(answers)

            # 少数クラスのサンプルだけ強い aug を使う（strong_transform がある時のみ）
            if (
                self.strong_transform is not None
                and mode_answer_idx in self.minority_answer_idx
            ):
                image = self.strong_transform(image)
            else:
                image = self.transform(image)

            return {"image": image, "question": question,"answers": torch.Tensor(answers), "mode_answer": int(mode_answer_idx),}
        else:
            image = self.transform(image)
            return {"image": image, "question": question}

    def __len__(self):
        return len(self.df)

## 5. 評価指標

In [ ]:
def VQA_criterion(batch_pred, batch_answers):
    """
    VQA タスクに用いられる評価関数．
    """
    total_acc = 0.

    for pred, answers in zip(batch_pred, batch_answers):
        acc = 0.
        for i in range(len(answers)):
            num_match = 0
            for j in range(len(answers)):
                if i == j:
                    continue
                if pred == answers[j]:
                    num_match += 1
            acc += min(num_match / 3, 1)
        total_acc += acc / 10

    return total_acc / len(batch_pred)

def vqa_acc_string(pred_str, gt_strings):
    """1サンプルの VQA accuracy（文字列ベース, leave-one-out）。"""
    acc = 0.0
    n = len(gt_strings)
    for i in range(n):
        num_match = sum(
            1 for j in range(n)
            if j != i and pred_str == gt_strings[j]
        )
        acc += min(num_match / 3, 1)
    return acc / n

def leaderboard_faithful_acc(preds, dataset, idx2answer):
    """リーダーボードと同じ採点での valid VQA accuracy。

    インデックス同士で照合する VQA_criterion は <unk> 同士が一致扱いになり
    valid を過大評価する（本番には <unk> ラベルが無いため）。これを避けるため:
      - 予測 idx → 文字列。<unk> は提出と同じく "unanswerable" に変換
      - GT は <unk> に潰さない「元の回答文字列」と照合
    こうすると valid が public とズレなくなる。

    Parameters
    ----------
    preds : list[int]
        dataset と同じ並び順の予測インデックス（valid は shuffle=False 前提）
    dataset : VQADataset
        元の回答文字列 dataset.df["answers"] を持つもの
    idx2answer : dict
    """

    total = 0.0
    for i, p in enumerate(preds):
        pred_str = idx2answer[p]
        if pred_str == UNK_ANSWER:
            pred_str = "unanswerable"
        gt_strings = [
            process_text(a["answer"]) for a in dataset.df["answers"][i]
        ]
        total += vqa_acc_string(pred_str, gt_strings)
    return total / len(preds)

## 6. モデル (ResNet)

In [ ]:
class BasicBlock(nn.Module):
    """
    ResNet の basic block
    """
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        """
        コンストラクタ．

        Parameters
        ----------
        in_channles: int
            入力のチャネル数
        out_channels:
            出力のチャネル数
        stride: int
            ストライド
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            ブロックへの入力

        Returns
        -------
        out: torch.Tensor
            ブロックへの出力
        """
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += self.shortcut(residual)
        out = self.relu(out)

        return out
    

class BottleneckBlock(nn.Module):
    """
    ResNet の bottleneck block
    """
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        """
        コンストラクタ．

        Parameters
        ----------
        in_channles: int
            入力のチャネル数
        out_channels:
            出力のチャネル数
        stride: int
            ストライド
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, stride=1)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            ブロックへの入力

        Returns
        -------
        out: torch.Tensor
            ブロックへの出力
        """
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out += self.shortcut(residual)
        out = self.relu(out)

        return out

class ResNet(nn.Module):
    """
    ResNet の実装
    """
    def __init__(self, block, layers):
        """
        コンストラクタ．

        Parameters
        ----------
        block: torch.nn.Module
            利用するブロックのクラス (BasicBlock / BottleneckBlock)
        layers: list
            各ブロックの層数
        """
        super().__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, layers[0], 64)
        self.layer2 = self._make_layer(block, layers[1], 128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], 256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], 512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.out_channels = 512 * block.expansion  # layer4 出力チャネル数
        self.fc = nn.Linear(512 * block.expansion, 512)

    def _make_layer(self, block, blocks, out_channels, stride=1):
        """
        同じ構成を繰り返す部分を生成する．

        Parameters
        ----------
        block: torch.nn.Module
            利用するブロックのクラス (BasicBlock / BottleneckBlock)
        blocks: int
            層数
        out_channels: int
            出力のチャネル数
        stride: int
            ストライド

        Returns
        -------
        layers: torch.nn.ModuleList
            生成した層
        """
        layers = []
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            入力データ

        Returns
        -------
        x: torch.Tensor
            ResNet によって生成される特徴量
        """
        x = self.forward_features(x)  # (B, out_channels, H', W')

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

    def forward_features(self, x):
        """avgpool/fc の前の空間特徴マップ (B, out_channels, H', W') を返す。
        cross-attention 融合で「画像のどこを見るか」を扱うために使う。"""
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        return x

def ResNet18():
    """
    ResNet18 を生成する関数．
    """
    return ResNet(BasicBlock, [2, 2, 2, 2])

def ResNet34():
    """
    ResNet34 を生成する関数．
    """
    return ResNet(BasicBlock, [3, 4, 6, 3])

def ResNet50():
    """
    ResNet50 を生成する関数．
    """
    return ResNet(BottleneckBlock, [3, 4, 6, 3])

## 7. モデル (VQAModel)

In [ ]:
# config の RESNET 文字列から ResNet 生成関数を引くためのテーブル
RESNET_FACTORY = {
    "resnet18": ResNet18,
    "resnet34": ResNet34,
    "resnet50": ResNet50,
}

class VQAModel(nn.Module):
    """
    VQA タスクを解くためのモデル．

    融合方式は config の FUSION で切替（"concat" / "cross_attention"）。
    """
    def __init__(self, vocab_size: int, n_answer: int, backbone: str = "resnet18",
                 fusion: str = None):
        """
        Parameters
        ----------
        vocab_size: int
            入力文の語彙数（dataset が渡す len(question2idx)+1）
        n_answer: int
            出力のクラス数
        backbone: str
            画像エンコーダの ResNet 種別 ("resnet18" / "resnet34" / "resnet50")
        fusion: str
            融合方式。None なら config の FUSION を使う。
        """
        super().__init__()
        if backbone not in RESNET_FACTORY:
            raise ValueError(
                f"Unknown backbone: {backbone} "
                f"(choose from {list(RESNET_FACTORY)})"
            )
        self.fusion = fusion or FUSION
        if self.fusion not in ("concat", "cross_attention"):
            raise ValueError(f"Unknown FUSION: {self.fusion}")

        self.resnet = RESNET_FACTORY[backbone]()  # avgpool+fc 経由で 512 次元

        # テキストエンコーダ: Embedding + 双方向LSTM。
        # question は単語インデックス列 (B, MAX_QLEN)。
        # dataset 側: UNK=vocab_size-1, PAD=vocab_size（=ここでの pad_idx）。
        self.pad_idx = vocab_size
        emb_dim = 300
        hidden = 256  # 双方向なので text 特徴は 2*hidden = 512 次元
        self.d = 512
        self.embedding = nn.Embedding(
            vocab_size + 1, emb_dim, padding_idx=self.pad_idx
        )
        self.lstm = nn.LSTM(
            emb_dim, hidden, batch_first=True, bidirectional=True
        )

        if self.fusion == "cross_attention":
            # 画像の空間特徴 (B, C, H, W) を d 次元トークン列に射影し、
            # 質問トークン(query)が画像トークン(key/value)に attention する。
            self.img_proj = nn.Conv2d(self.resnet.out_channels, self.d, kernel_size=1)
            self.cross_attn = nn.MultiheadAttention(
                embed_dim=self.d, num_heads=8, batch_first=True
            )
            self.attn_norm = nn.LayerNorm(self.d)

        # concat / cross_attention とも最終特徴は [質問512, 画像由来512] の連結。
        self.fc = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, n_answer)
        )

        # 補助ヘッド（#3「画像を捨てさせない学習」）。画像プール特徴(512)だけから
        # 回答を予測する。AUX_IMAGE_LOSS_WEIGHT>0 のときだけ作る。学習で
        # 画像branch に必ず勾配を流し、言語prior へのショートカットを防ぐ。
        # 推論では使わない（forward の return_aux=False で無視）。
        # 注意: train と inference で AUX_IMAGE_LOSS_WEIGHT の ≷0 を揃えること
        #       （FUSION と同様、アーキテクチャが変わるため。state_dict 整合）。
        self.aux_image_fc = (
            nn.Linear(self.d, n_answer) if AUX_IMAGE_LOSS_WEIGHT > 0 else None
        )

    def _encode_question(self, question):
        """質問 → (トークン系列 (B,L,512), マスク (B,L,1), プール特徴 (B,512))。"""
        q = question.long()
        mask = (q != self.pad_idx).unsqueeze(-1).float()  # (B, L, 1)
        lstm_out, _ = self.lstm(self.embedding(q))  # (B, L, 512)
        pooled = (lstm_out * mask).sum(1) / mask.sum(1).clamp(min=1.0)  # (B, 512)
        return lstm_out, mask, pooled

    def forward(self, image, question, return_aux=False):
        q_tokens, q_mask, q_pooled = self._encode_question(question)

        # backbone は1回だけ通し、空間特徴とプールベクトルの両方を得る
        # （concat/cross_attention/補助ヘッドで共有。二重forwardを避ける）。
        feat_map = self.resnet.forward_features(image)         # (B, C, H, W)
        pooled = self.resnet.avgpool(feat_map).flatten(1)      # (B, C)
        img_vec = self.resnet.fc(pooled)                       # (B, 512) = resnet(image)

        if self.fusion == "concat":
            # 画像を1ベクトル(512)に潰して質問プール特徴と連結（従来方式）。
            x = torch.cat([img_vec, q_pooled], dim=1)

        else:  # cross_attention
            feat = self.img_proj(feat_map)  # (B, d, H, W)
            img_tokens = feat.flatten(2).transpose(1, 2)  # (B, HW, d)

            # 質問トークン(query)が画像トークン(key/value)に attention。
            # 画像トークンは全て有効なので key_padding_mask は不要。
            # PAD 質問位置は後段のプールで mask により除外する。
            attended, _ = self.cross_attn(
                query=q_tokens, key=img_tokens, value=img_tokens,
            )
            attended = self.attn_norm(attended)  # (B, L, d)
            # PAD 質問位置を除いて平均 → 画像に接地した質問特徴 (B, d)
            grounded = (attended * q_mask).sum(1) / q_mask.sum(1).clamp(min=1.0)
            x = torch.cat([grounded, q_pooled], dim=1)

        logits = self.fc(x)
        if return_aux:
            # 画像プール特徴だけからの予測（#3 補助ロス用）。
            aux = self.aux_image_fc(img_vec) if self.aux_image_fc is not None else None
            return logits, aux
        return logits

## 8. 損失関数

In [ ]:
class SoftCrossEntropyLoss(nn.Module):

    def __init__(self, weight=None):
        super().__init__()
        # クラス重み (num_classes,)。None なら均等。
        self.register_buffer("weight", weight)

    def forward(self, logits, soft_targets):

        log_probs = F.log_softmax(logits, dim=1)

        if self.weight is not None:
            # クラスごとに重み付け（ターゲット分布の各クラスのlossをスケール）
            log_probs = log_probs * self.weight

        loss = -(soft_targets * log_probs).sum(dim=1)

        return loss.mean()
    
def build_soft_target(answers, num_classes, ignore_index=None):
    """
    answers:
        (batch_size, 10)

    return:
        (batch_size, num_classes)
    """

    batch_size = answers.shape[0]

    target = torch.zeros(
        batch_size,
        num_classes,
        device=answers.device,
    )

    for i in range(batch_size):
        valid_count = 0

        for ans in answers[i]:
            ans_idx = int(ans)

            if ignore_index is not None and ans_idx == ignore_index:
                continue

            target[i, ans_idx] += 1
            valid_count += 1

        # 全て ignore 対象なら元の分布でフォールバック
        if valid_count == 0:
            for ans in answers[i]:
                target[i, int(ans)] += 1
            valid_count = answers.shape[1]

        target[i] /= valid_count

    return target

## 9. 学習・検証ループ

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device, unanswerable_idx=None, desc="train"):
    model.train()

    total_loss = 0
    total_acc = 0
    total_simple_acc = 0

    start = time.time()

    pbar = tqdm(dataloader, desc=desc, leave=True)
    for batch in pbar:
        image = batch["image"].to(device)
        question = batch["question"].to(device)
        answers = batch["answers"].to(device)
        mode_answer = batch["mode_answer"].to(device)

        pred = model(image, question)

        if LOSS_TYPE == "hard":
            loss = criterion(pred, mode_answer)
        elif LOSS_TYPE == "soft":
            soft_target = build_soft_target(answers, pred.shape[1], ignore_index=unanswerable_idx)
            loss = criterion(pred, soft_target)
        else:
            raise ValueError(f"Unknown LOSS_TYPE: {LOSS_TYPE}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)
        total_simple_acc += (pred.argmax(1) == mode_answer).float().mean().item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")

    n = len(dataloader)
    return total_loss / n, total_acc / n, total_simple_acc / n, time.time() - start


@torch.no_grad()
def validate(model, dataloader, criterion, device, unanswerable_idx=None, desc="valid"):
    model.eval()

    total_loss = 0
    total_acc = 0
    total_simple_acc = 0

    start = time.time()

    for batch in tqdm(dataloader, desc=desc, leave=True):
        image = batch["image"].to(device)
        question = batch["question"].to(device)
        answers = batch["answers"].to(device)
        mode_answer = batch["mode_answer"].to(device)

        pred = model(image, question)

        if LOSS_TYPE == "hard":
            loss = criterion(pred, mode_answer)
        elif LOSS_TYPE == "soft":
            soft_target = build_soft_target(answers, pred.shape[1], ignore_index=unanswerable_idx)
            loss = criterion(pred, soft_target)

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)
        total_simple_acc += (pred.argmax(1) == mode_answer).float().mean().item()

    n = len(dataloader)
    return total_loss / n, total_acc / n, total_simple_acc / n, time.time() - start


## 10. データの準備

学習データ・検証データ・テストデータを読み込む。辞書は学習データのものを検証・テストへ反映する。

In [ ]:
set_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    raise RuntimeError(
        "GPU (CUDA/MPS) が利用できません。CPU 実行は許可されていません。"
    )
print("device =", device)

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = VQADataset(
    df_path="./data/train_split.json",
    image_dir="./data/train",
    transform=transform,
)

valid_dataset = VQADataset(
    df_path="./data/valid_split.json",
    image_dir="./data/train",
    transform=transform,
)
valid_dataset.update_dict(train_dataset)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True,
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)


## 11. 学習

検証 Acc が最良のモデルを `model.pt` として保存する。

In [ ]:
os.makedirs("./outputs/checkpoints", exist_ok=True)

model = VQAModel(
    vocab_size=len(train_dataset.question2idx) + 1,
    n_answer=len(train_dataset.answer2idx),
    backbone=RESNET,
).to(device)
unanswerable_idx = train_dataset.answer2idx.get("unanswerable")

if LOSS_TYPE == "hard":
    criterion = nn.CrossEntropyLoss()
elif LOSS_TYPE == "soft":
    criterion = SoftCrossEntropyLoss()
else:
    raise ValueError(f"Unknown LOSS_TYPE: {LOSS_TYPE}")

if OPTIMIZER == "adam":
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY,
    )
else:
    raise ValueError(f"Unknown OPTIMIZER: {OPTIMIZER}")

best_acc = -1.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, train_simple_acc, train_time = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
        unanswerable_idx=unanswerable_idx,
        desc=f"train [{epoch+1}/{NUM_EPOCHS}]",
    )
    valid_loss, valid_acc, valid_simple_acc, valid_time = validate(
        model, valid_loader, criterion, device,
        unanswerable_idx=unanswerable_idx,
        desc=f"valid [{epoch+1}/{NUM_EPOCHS}]",
    )
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Acc={train_acc:.4f} Valid Acc={valid_acc:.4f}"
    )
    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(model.state_dict(), "./model.pt")

print("best valid acc =", best_acc)


## 12. 推論 (テストデータ)

`data/valid.json` をテストデータとして予測し `submission.npy` を作成する。

In [ ]:
test_dataset = VQADataset(
    df_path="./data/valid.json",
    image_dir="./data/valid",
    transform=transform,
    answer=False,
)
test_dataset.update_dict(train_dataset)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=2, pin_memory=True,
)

model.load_state_dict(torch.load("./model.pt", map_location=device))
model.eval()

submission = []
with torch.no_grad():
    for batch in tqdm(test_loader, desc="inference"):
        image = batch["image"].to(device)
        question = batch["question"].to(device)
        pred = model(image, question)
        pred = pred.argmax(1).item()
        submission.append(train_dataset.idx2answer[pred])

submission = np.array(submission)
np.save("./submission.npy", submission)
print("submission.npy saved:", submission.shape)


## 13. 提出ファイル作成

`submission.npy` / `model.pt` / 本 Notebook をまとめて zip 化する。
**注意**: `NOTEBOOK_NAME` を実際に保存している Notebook のファイル名に合わせること。

In [ ]:
# 本 Notebook のファイル名（保存名に合わせて変更）
NOTEBOOK_NAME = "DL_Basic_2026_Spring_competition_VQA.ipynb"

with ZipFile("submission.zip", "w") as zf:
    zf.write("submission.npy")
    zf.write("model.pt")
    if os.path.exists(NOTEBOOK_NAME):
        zf.write(NOTEBOOK_NAME)
    else:
        print(f"[警告] {NOTEBOOK_NAME} が見つかりません。"
              f" Notebook を保存後にこのセルを再実行してください。")

print("submission.zip created")
